# 10: Foundation-model embeddings

Runs a pretrained backbone, frozen and without any training on retinal images, once over every
processed image (all splits, train included) and saves one embedding per image. A linear probe
is then fitted locally on the train embeddings (`python -m retinaguard.train.linear_probe`), and
the embeddings are a reference for drift monitoring.

**Settings** (right-hand panel): Accelerator **GPU T4 x1**, Internet **On** (downloads the
pretrained weights).
**Inputs** (*Add Input*):
- *Notebooks* → *Your Work* → **01**, **02**, **03**, **04** (processed image shards)
- *Datasets* → *Your Datasets* → **retinaguard-manifest**

**Run:** set `MODEL` below, then *Save Version* → *Save & Run All (Commit)*. About 20–30 min
for DINOv2 ViT-B.

**Output** (`/kaggle/working/embeddings/<model>/`): `embeddings.npy`, `index.parquet`,
`meta.json`.

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
MODEL = "dinov2_vitb"  # a model config in configs/model/
LOCAL_MEAN_NORM = False  # natural-image backbones expect natural colours

In [ ]:
# Kaggle's preinstalled albumentations may predate the 2.x API the package imports.
EXTRAS = '"pandera>=0.20" "hydra-core>=1.3" "timm>=1.0" "albumentations>=2.0"'
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" {EXTRAS}

In [ ]:
# --- 1. Inputs -------------------------------------------------------------------
import re
import subprocess
from pathlib import Path

import torch

from retinaguard import __version__

INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/embeddings") / MODEL
manifests = sorted(INPUT_ROOT.glob("**/manifest.parquet"))
assert manifests, "manifest.parquet not found: attach the retinaguard-manifest dataset"
shard = re.compile(r"^(eyepacs|aptos|idrid|ddr)_\d{3}\.zip$")
shard_dirs = sorted({p.parent for p in INPUT_ROOT.glob("**/*.zip") if shard.match(p.name)})
sources = sorted({p.name.split("_")[0] for d in shard_dirs for p in d.glob("*.zip")})
assert sources == ["aptos", "ddr", "eyepacs", "idrid"], f"shards found for {sources} only"

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("	")[0]
assert torch.cuda.is_available(), (
    f"no GPU (torch {torch.__version__}): set Accelerator to GPU in the notebook settings, "
    "then save a new version"
)
print(f"retinaguard {__version__} @ {git_sha[:10]}, GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# --- 2. Extract ------------------------------------------------------------------
import time

from retinaguard.train.extract_embeddings import main as extract

start = time.perf_counter()
extract(
    [
        f"--model={MODEL}",
        f"--manifest={manifests[0]}",
        "--shard-dirs",
        *map(str, shard_dirs),
        f"--out-dir={OUT_DIR}",
        "--batch-size=32",  # ViT-B at 512 px: 1,369 tokens per image
        *(["--local-mean-norm"] if LOCAL_MEAN_NORM else []),
    ]
)
print(f"{(time.perf_counter() - start) / 60:.1f} min")

**After the version finishes:** download `embeddings.npy`, `index.parquet` and `meta.json` from
the **Output** tab into `data/interim/embeddings/<model>/`.